# AF2 frozen parent + IGEM feature residual
Arm `AF2IGEM1`. Hanya residual klasifikasi IGEM yang dilatih; AF2 parent dibekukan. Validation-only, tanpa test.

In [ ]:
import torch
assert torch.cuda.is_available(), 'STOP CEPAT: aktifkan Runtime > Change runtime type > T4 GPU, lalu Run all.'
print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
ARM='AF2IGEM1'
BRANCH='codex/af2-parent-residual'
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, sys, tarfile, time, json
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    result=subprocess.run(clone)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    time.sleep(2)
else: raise RuntimeError('Git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT=resolve_drive_project_root(required_relative_paths=('bundles/faruq-development-v3-grouped.tar','experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt'))
ARCHIVE=require_project_artifact(PROJECT,'bundles/faruq-development-v3-grouped.tar'); AF2=require_project_artifact(PROJECT,'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt')
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
GROUPED=DATA/'faruq_grouped_summary.json'; assert (DATA/'data.yaml').is_file() and GROUPED.is_file() and not (DATA/'test').exists()
OUTPUT=PROJECT/'experiments/faruq-v3-af2-parent-residual-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
print('ARM:',ARM,'| PROJECT:',PROJECT,'| AF2:',AF2)

In [ ]:
from coffee_detector.af2_parent_residual import run_af2_parent_residual_static_audit
STATIC=OUTPUT/f'static_audit_{ARM}.json'; audit=run_af2_parent_residual_static_audit(AF2,STATIC,device='cpu',image_size=64)
print('STATIC:',audit['decision'],STATIC); assert audit['decision']=='PASS','STOP: static audit gagal; jangan training.'

In [ ]:
LOG=OUTPUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_parent_residual_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream:
    process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    while process.poll() is None:
        time.sleep(60); lines=LOG.read_text(errors='replace').splitlines(); status=[line for line in lines if line.startswith('AF2-')]
        if status: print(status[-1],flush=True)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUTPUT/'val_reports'/f'{ARM}_seed42_result.json'; print(json.dumps(json.loads(RESULT.read_text()),indent=2,ensure_ascii=False))